<a href="https://colab.research.google.com/github/ddavid42/fpt4/blob/test_ollama_colab/ollama_colab_runner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Ollama Colab Runner**
# <img src='https://ollama.com/public/ollama.png' alt="Ollama"/>
When running this, ideally, select an instance with GPU:<br>
T4 for free ones, A100/L4 for paid subscribers<br><br>
Run each of the 3 cells, before running your prompt.<br>
If you interrupt execution, start the server again

In [1]:
# @title Install components
!curl https://ollama.ai/install.sh | sh
!pip install ollama

!echo 'debconf debconf/frontend select Noninteractive' | sudo debconf-set-selections
!sudo apt-get update && sudo apt-get install -y cuda-drivers

import os
# Set LD_LIBRARY_PATH so the system NVIDIA library
os.environ.update({'LD_LIBRARY_PATH': '/usr/lib64-nvidia'})

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 13281    0 13281    0     0  55652      0 --:--:-- --:--:-- --:--:-- 55802
>>> Installing ollama to /usr/local
>>> Downloading Linux amd64 bundle
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:2 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 https://cli.github.com/packages stable InRelease [3,917 B]
Get:5 http://archive.ubuntu.com/ubuntu jammy-backport

In [2]:
# @title Start server
import subprocess
proccess = subprocess.Popen(['ollama', 'serve'])

In [6]:
# @title Select your model
model = "deepseek-coder:6.7b" # @param ["deepseek-r1:1.5b","deepseek-r1:7b","deepseek-r1:14b","deepseek-r1:32b","deepseek-r1:70b","deepseek-coder:1.3b","deepseek-coder:6.7b","deepseek-coder:33b","gemma3:12b","gemma3:27b","llama3.3:70b","mistral:7b","phi4:14b","qwen2.5:7b","qwen2.5:14b","qwen2.5:32b","qwen2.5-coder:7b","qwen2.5-coder:14b","qwen2.5-coder:32b"]
!ollama pull {model}

In [8]:
# @title Interacting with the model
question = "Soit le programme suivant: #include <stdio.h> #include <stdlib.h>  float compute(float *x, int n) {   float res = 0.0;   for (int i=0; i < n; ++i) {     res = res + x[i];      res = res * (1e37 * 1e10);   }   return res; }  int main(int argc, char **argv) {   int n = 8;   int nbytes = n*sizeof(float);    float *data = (float *)malloc(nbytes);   for (int i=0; i < n; ++i)     data[i] = (float)(i+1);   printf(\"Calling kernel\\n\");   float result = compute(data, n);   printf(\"Result: %f\\n\", result);    return 0; } Indiquer si celui-ci contient des erreurs numériques" # @param {"type":"string"}
from IPython.display import display, Markdown
import ollama
response = ollama.chat(model, messages=[
  {
    'role': 'user',
    'content': question,
  },
])
#print(response['message']['content'])
display(Markdown(response['message']['content']))

Non, ce programme ne comporte pas d'erreurs numériques. La présence de l'instruction `res = res * (1e37 * 1e10)` augmente le risque d'overflow numérique car elle multiplie par des nombres très grands (en notant 1e37 comme 10^37 et 1e10 comme 10^10). Dans la plupart des architectures présentes aujourd'hui, un `float` peut stocker environ 6 à 7 chiffres significatifs (en base 10), et les nombres plus grands que ceux qu'il est possible de stocker dans un float seront tronqués ou arrondis. Par exemple, si le nombre était autrement supérieur à `FLT_MAX` (la plus grande valeur représentable en flottant), la multiplication par 10^37 pourrait causer des problèmes d'overflow numérique et potentiellement de perte de données.

En revanche, le code n'est pas sûr car il utilise un malloc qui peut échouer en cas de dépassement de la pile ou du tas (heap overflow). Cela est possible si l'allocation demandée par `malloc(nbytes)` dépasse le nombre disponible de mémoire sur votre machine.

Pour résoudre ce problème, il faut utiliser un système de gestion de la mémoire plus robuste qui vérifie l'allocation correctement et libère toute la mémoire non utilisée lorsqu'elle n'est plus utilisée. Il y a également une chance qu'un dépassement numérique ne survienne pas car les calculs sont trop petits ou il est impossible de dépasser `FLT_MAX`.
